# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [ ]:
# 1. Инициализация окружения Colab и подтягивание свежих скриптов из репозитория
!nvidia-smi
import os, sys
if not os.path.exists("_tools/colab_neural_trainer.py"):
    !git clone --depth 1 https://github.com/BulletproofBoris/crypto_bot.git _repo_tmp 2>/dev/null || true
    !cp -r _repo_tmp/_tools .
sys.path.insert(0, os.path.abspath("."))
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Модули _tools.colab_models и _tools.colab_neural_trainer готовы к работе!")

In [ ]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (редактируемые параметры)
# Асинхронная сетка окон: H=25 -> 2 окна (375k), H=12 -> 5 окон (210k), H=5 -> 12 окон (75k)
HORIZON = 25                                               # Горизонт H (сек): 5, 12 или 25
WINDOWS = "all"                                            # Список окон: "all" или "0,1"
ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"   # Архитектуры через запятую
RUNS_PER_ARCH = 10                                         # Число независимых ранов на архитектуру
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 2e-3                                                  # Скорость обучения AdamW

df_leaderboard = run_colab_pipeline(
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR
)

In [ ]:
# 3. Альтернативный запуск через CLI с аргументами командной строки и выгрузка архива
# !python -m _tools.colab_neural_trainer --horizon 12 --windows 0,1,2,3,4 --runs 10 --epochs 10 --batch-size 2048
ZIP_NAME = f"prl_ensemble_pack_h{HORIZON}s.zip"
if os.path.exists(ZIP_NAME):
    !curl -s -F "reqtype=fileupload" -F "time=24h" -F "fileToUpload=@{ZIP_NAME}" https://litterbox.catbox.moe/resources/internals/api.php
    try:
        from google.colab import files
        files.download(ZIP_NAME)
    except Exception:
        pass